# Cost Estimates for 3-D Disks

In [1]:
%load_ext autoreload
%autoreload 2
%pylab inline

Populating the interactive namespace from numpy and matplotlib


In [2]:
def days_per_experiment(log_step=0.03,nz=3,n=16,cores_per_node=48,
        verbose=0,nodes=4,inner_orbits=200,radial_extent=3,orbit_AU=10,
        c_1AU=1,v_1AU=30,T_power=0.25,particles_per_bin=10,bins=3,
        courant=0.25,microsec_per_cell=0.6,microsec_per_particle=0.07):

    ny=2*pi/log_step
    nx=int(np.log(radial_extent)/log_step)
    nupdate=0
    for ix in range(nx):
        r=orbit_AU*exp(ix*log_step)
        v_orbit=v_1AU*1e5/r**0.5
        c_sound=c_1AU*1e5/r**T_power
        ds=r*log_step/n
        dt=courant*ds*1.5e13/c_sound
        if ix==0:
            t_orbit=3e7*r**1.5
            if verbose>1:
                nt=int(t_orbit/dt)
                print(c_sound/1e5,v_orbit/1e5,ds,nt)
        nt=int(t_orbit/dt)
        n_step=nt*ny*nz*n**3
        nupdate+=n_step
        if verbose>1 and ix%10==0:
            print('{:3d}{:5d}{:10.2e}'.format(ix,nt,n_step))
        
    particles=particles_per_bin*bins
    microsec_per_step=microsec_per_cell+particles*microsec_per_particle
    if verbose>0:
        print('{:.2f} microsec/cell-update assumed'.format(microsec_per_step))
    core_hours_per_orbit=nupdate*microsec_per_step*1e-6/3600.
    cores=cores_per_node*nodes
    hours_per_orbit=core_hours_per_orbit/cores
    hours=hours_per_orbit*inner_orbits
    Mch=hours*cores*1e-6
    print('requires about {:.1f} days and {:.3f} Mch on {} cores'.format(hours/24,Mch,cores))
days_per_experiment()

requires about 1.3 days and 0.006 Mch on 192 cores


In [3]:
days_per_experiment(log_step=0.032,nz=9,inner_orbits=1.25e6,orbit_AU=10,radial_extent=10,nodes=100,particles_per_bin=6,bins=1)

requires about 379.3 days and 43.696 Mch on 4800 cores
